In [28]:
# Basic data processing
import numpy as np
import pandas as pd
import os

# Machine learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import fbeta_score, confusion_matrix

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

In [29]:
# Load training set
train_df = pd.read_csv('/bohr/train-ac7m/v2/train.csv')
print(f"The size of training set: {len(train_df)}")
train_df.head()

In [30]:
import pandas as pd
import numpy as np

def feature_engineering(X):
    # 1. 将 -9999 替换为 NaN
    X.replace(-9999, np.nan, inplace=True)

    # 2. 缺失值处理（不处理）

    # 3. 分箱
    for col in ['modelMag_u', 'modelMag_g', 'modelMag_r', 'modelMag_i', 'modelMag_z']:
        X[f'{col}_bin'] = pd.qcut(X[col], q=4, labels=False)  # 4个箱子

    # 4. 颜色指数
    X['c_gr'] = X['modelMag_g'] - X['modelMag_r']
    X['c_ri'] = X['modelMag_r'] - X['modelMag_i']
    X['c_iz'] = X['modelMag_i'] - X['modelMag_z']
    X['c_gi'] = X['modelMag_g'] - X['modelMag_i']
    X['c_uz'] = X['modelMag_u'] - X['modelMag_z']

    # 5. 特征组合（平方和交互特征）
    X['u2'] = X['modelMag_u'] ** 2
    X['g2'] = X['modelMag_g'] ** 2
    X['r2'] = X['modelMag_r'] ** 2
    X['i2'] = X['modelMag_i'] ** 2
    X['z2'] = X['modelMag_z'] ** 2
    
    X['ug'] = X['modelMag_u'] * X['modelMag_g']
    X['gr'] = X['modelMag_g'] * X['modelMag_r']
    X['ri'] = X['modelMag_r'] * X['modelMag_i']
    X['iz'] = X['modelMag_i'] * X['modelMag_z']
    X['ur'] = X['modelMag_u'] * X['modelMag_r']
    X['gi'] = X['modelMag_g'] * X['modelMag_i']
    X['ui'] = X['modelMag_u'] * X['modelMag_i']
    X['gz'] = X['modelMag_g'] * X['modelMag_z']
    X['rz'] = X['modelMag_r'] * X['modelMag_z']

    # 6. 统计特征
    X['mean'] = X[['modelMag_u', 'modelMag_g', 'modelMag_r', 'modelMag_i', 'modelMag_z']].mean(axis=1)
    X['std'] = X[['modelMag_u', 'modelMag_g', 'modelMag_r', 'modelMag_i', 'modelMag_z']].std(axis=1)
    X['max'] = X[['modelMag_u', 'modelMag_g', 'modelMag_r', 'modelMag_i', 'modelMag_z']].max(axis=1)
    X['min'] = X[['modelMag_u', 'modelMag_g', 'modelMag_r', 'modelMag_i', 'modelMag_z']].min(axis=1)
    X['range'] = X['max'] - X['min']

    # 7. 颜色指数的平方
    X['c_gr2'] = X['c_gr'] ** 2
    X['c_ri2'] = X['c_ri'] ** 2
    X['c_iz2'] = X['c_iz'] ** 2
    X['c_gi2'] = X['c_gi'] ** 2
    X['c_uz2'] = X['c_uz'] ** 2

    # 8. 其他多项式特征
    X['u3'] = X['modelMag_u'] ** 3
    X['g3'] = X['modelMag_g'] ** 3
    X['r3'] = X['modelMag_r'] ** 3
    X['i3'] = X['modelMag_i'] ** 3
    X['z3'] = X['modelMag_z'] ** 3

    # 9. 交互特征的平方
    X['ug2'] = X['ug'] ** 2
    X['gr2'] = X['gr'] ** 2
    X['ri2'] = X['ri'] ** 2
    X['iz2'] = X['iz'] ** 2
    X['ur2'] = X['ur'] ** 2
    X['gi2'] = X['gi'] ** 2
    X['ui2'] = X['ui'] ** 2
    X['gz2'] = X['gz'] ** 2
    X['rz2'] = X['rz'] ** 2

    # 10. 颜色指数的交互特征
    X['c_gr_ri'] = X['c_gr'] * X['c_ri']
    X['c_ri_iz'] = X['c_ri'] * X['c_iz']
    X['c_gi_uz'] = X['c_gi'] * X['c_uz']

    # 11. 计算颜色指数的绝对值
    X['abs_c_gr'] = np.abs(X['c_gr'])
    X['abs_c_ri'] = np.abs(X['c_ri'])
    X['abs_c_iz'] = np.abs(X['c_iz'])
    X['abs_c_gi'] = np.abs(X['c_gi'])
    X['abs_c_uz'] = np.abs(X['c_uz'])

    # 12. 计算特征的对数（避免负值）
    for col in ['modelMag_u', 'modelMag_g', 'modelMag_r', 'modelMag_i', 'modelMag_z']:
        X[f'log_{col}'] = np.log1p(X[col])  # 使用log1p以避免对0取对数

    # 13. 计算特征的平方根
    for col in ['modelMag_u', 'modelMag_g', 'modelMag_r', 'modelMag_i', 'modelMag_z']:
        X[f'sqrt_{col}'] = np.sqrt(X[col].clip(lower=0))  # clip避免负值

    # 14. 选择特征
    feature_cols = ['ra', 'dec'] + \
                   [f'{col}_bin' for col in ['modelMag_u', 'modelMag_g', 'modelMag_r', 'modelMag_i', 'modelMag_z']] + \
                   ['c_gr', 'c_ri', 'c_iz', 'c_gi', 'c_uz',
                    'u2', 'g2', 'r2', 'i2', 'z2',
                    'ug', 'gr', 'ri', 'iz', 'ur', 'gi', 'ui', 'gz', 'rz',
                    'mean', 'std', 'max', 'min', 'range',
                    'c_gr2', 'c_ri2', 'c_iz2', 'c_gi2', 'c_uz2',
                    'u3', 'g3', 'r3', 'i3', 'z3',
                    'ug2', 'gr2', 'ri2', 'iz2', 'ur2', 'gi2', 'ui2', 'gz2', 'rz2',
                    'c_gr_ri', 'c_ri_iz', 'c_gi_uz',
                    'abs_c_gr', 'abs_c_ri', 'abs_c_iz', 'abs_c_gi', 'abs_c_uz'] + \
                   [f'log_{col}' for col in ['modelMag_u', 'modelMag_g', 'modelMag_r', 'modelMag_i', 'modelMag_z']] + \
                   [f'sqrt_{col}' for col in ['modelMag_u', 'modelMag_g', 'modelMag_r', 'modelMag_i', 'modelMag_z']]

    return X[feature_cols]


In [31]:
feature_cols = ['ra', 'dec', 'modelMag_u', 'modelMag_g', 'modelMag_r', 'modelMag_i', 'modelMag_z']
X_train = feature_engineering(train_df[feature_cols]).values
y_train = train_df['type'].values

print(f"Size: {X_train.shape}")
print(f"Label Distribution:\n{pd.Series(y_train).value_counts()}")

In [33]:
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import fbeta_score
from sklearn.preprocessing import LabelEncoder
import numpy as np

# 假设你已经有了 X_train 和 y_train

# 标签编码
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)

# 计算每个类别的样本权重
class_weights = {
    0: 1,    # GALAXY
    1: 4.67, # STAR
    2: 4.67  # QSO
}

# 创建样本权重数组
sample_weights = np.array([class_weights[label] for label in y_train_encoded])

# 定义 F2 分数
def f2_score(y_true, y_pred):
    return fbeta_score(y_true, y_pred, beta=2, average='weighted')

# 创建 XGBoost 分类器
model = xgb.XGBClassifier(
    use_label_encoder=False,  # 不使用标签编码器
    eval_metric='mlogloss',   # 评估指标
    n_estimators=1000,         # 树的数量
    learning_rate=0.2,        # 学习率
    max_depth=6,              # 树的最大深度
    min_child_weight=1,       # 最小子权重
    gamma=0,                  # 节点分裂所需的最小损失减少
    subsample=1,              # 训练样本的子采样比率
    colsample_bytree=0.8,       # 每棵树的特征子采样比率
    colsample_bylevel=0.8,      # 每个级别的特征子采样比率
    reg_alpha=0.1,              # L1 正则化项系数
    reg_lambda=1.5,             # L2 正则化项系数
    objective='multi:softmax',# 目标函数
    num_class=3,              # 类别数（对于多分类问题）
    random_state=42,          # 随机种子
    n_jobs=-1                 # 使用所有可用的 CPU 核心
)

# 5折交叉验证
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 存储每折的得分
f2_scores = []

for train_index, test_index in kf.split(X_train):
    X_train_fold, X_test_fold = X_train[train_index], X_train[test_index]
    y_train_fold, y_test_fold = y_train_encoded[train_index], y_train_encoded[test_index]
    
    # 创建样本权重数组
    sample_weights_fold = np.array([class_weights[label] for label in y_train_fold])
    
    # 训练模型，传递样本权重
    model.fit(X_train_fold, y_train_fold, sample_weight=sample_weights_fold)
    
    # 进行预测
    y_pred_fold = model.predict(X_test_fold)
    
    # 计算 F2 分数
    score = f2_score(y_test_fold, y_pred_fold)
    f2_scores.append(score)

# 输出每折的得分和平均得分
print("每折得分:", f2_scores)
print("平均得分:", np.mean(f2_scores))

# 计算训练集上的 F2 分数
y_train_pred = model.predict(X_train)
# 将预测结果转换回原始标签
y_train_pred_labels = label_encoder.inverse_transform(y_train_pred)

# 计算 F2 分数
f2_train = fbeta_score(y_train, y_train_pred_labels, beta=2, average='weighted')
print(f"F2-score on training set: {f2_train:.4f}")

# 现在进行最终预测
# 假设你有一个新的数据集 X_test 需要进行预测
# X_test = ...  # 你的测试数据

# 进行预测
# y_pred_final = model.predict(X_test)

# 如果需要将预测结果转换回原始标签
# y_pred_final_labels = label_encoder.inverse_transform(y_pred_final)

# 输出预测结果
# print("预测结果:", y_pred_final_labels)


In [ ]:
if os.environ.get('DATA_PATH'):
    DATA_PATH = os.environ.get("DATA_PATH") + "/"  
else:
    print("When baseline is running, this error message will appear because the test set cannot be read, which is a normal phenomenon.")

# DATA_PATH = "./"
# Load validation set and calculate the public score on leaderboard A
A_df = pd.read_csv(DATA_PATH + 'valdata.csv')

# Get features
X_A = feature_engineering(A_df[feature_cols]).values


# Predict
y_A_pred = label_encoder.inverse_transform(model.predict(X_A))

# Generate submission file
submit_A = pd.DataFrame({
    'objid': A_df['objid'],
    'type': y_A_pred
})

# Save
submit_A.to_csv('./submission_val.csv', index=False)
print("submission_val.csv has been saved")
submit_A.head()

In [ ]:
# Load test set and calculate the public score on leaderboard B
B_df = pd.read_csv(DATA_PATH + 'testdata.csv')

# Get features
X_B = feature_engineering(B_df[feature_cols]).values

# Predict
y_B_pred = label_encoder.inverse_transform(model.predict(X_B))

# Generate submission file
submit_B = pd.DataFrame({
    'objid': B_df['objid'],
    'type': y_B_pred
})

# Save
submit_B.to_csv('./submission_test.csv', index=False)
print("submission_test.csv has been saved")
submit_B.head()

In [ ]:
import zipfile
# Define the files to be packaged and the compressed file name.
files_to_zip = ['submission_val.csv', 'submission_test.csv']
zip_filename = 'submission.zip'

# Create a zip file.
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        # Add files to the zip file.
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} is created succefully!')